# 5장 01. 최종 확인 자료 모으기

이 notebook은 최종 확인에 필요한 report artifact가 준비되어 있는지 확인합니다.


## 기본 개념: release evidence packet

Release evidence packet은 배포 결정을 위해 필요한 근거 묶음입니다. MLOps에서는 모델 metric 하나만으로 release를 결정하지 않습니다. 데이터 조건, 모델 평가 기록, serving 배포 상태, 운영 관측, rollback 가능성을 함께 봐야 합니다.

각 artifact는 다른 질문에 답합니다. 데이터 리포트는 평가 입력이 믿을 만한지, metric 기록은 후보 모델이 어떤 조건에서 평가됐는지, GitOps/KServe 확인 결과는 배포 경로가 준비됐는지, 운영 로그와 metric은 배포 후 관측 가능한지 보여 줍니다.

좋은 release evidence는 “통과/실패”만 적지 않습니다. 어떤 근거가 확인됐고, 어떤 근거가 미확인이고, 누가 무엇을 확인해야 다음 단계로 갈 수 있는지 남깁니다.

| Evidence | 답하는 질문 | 다음 단계 연결 |
| --- | --- | --- |
| data quality report | 평가 데이터가 사용 가능한가 | model evaluation |
| metric record | 후보 모델이 어떤 조건에서 평가됐는가 | MLflow registry, release gate |
| serving inspection | candidate가 배포 경로에 연결됐는가 | Argo CD, KServe |
| observability report | 운영 중 추적 가능한가 | incident response |
| release approval | 기준과 미확인 리스크가 정리됐는가 | approve, hold, rollback |

이 노트북은 자동 승인 도구가 아닙니다. 여러 artifact가 어떤 역할을 하는지 확인하고, 최종 report에서 근거를 빠뜨리지 않게 만드는 정리 단계입니다.


In [ ]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


In [ ]:
from pathlib import Path

reports = [
    "artifacts/reports/drift_report.md",
    "artifacts/reports/quality_issue_trace.md",
    "artifacts/reports/release_approval.md",
    "artifacts/reports/ai_qa_checklist.md",
]

rows = []
for report in reports:
    path = Path(report)
    if not path.exists():
        path = Path("..") / report
    if not path.exists():
        path = Path("../..") / report
    rows.append({"report": report, "exists": path.exists(), "path_used": str(path)})

pd.DataFrame(rows)


## 1. 각 파일의 첫 줄 보기

파일이 있으면 첫 줄만 확인합니다. 전체 보고서 작성은 Markdown을 읽고 직접 정리합니다.


In [ ]:
for row in rows:
    if row["exists"]:
        text = Path(row["path_used"]).read_text()
        print(row["report"], "=>", text.splitlines()[0])
